# Topic-to-Video Generation

This notebook contains the full, working pipeline: an orchestrated task graph
that takes a topic or URL and produces a 30-40 second, 1080x1920 vertical
video with narration and captions.

See `APPROACH.md` in the project root for the full design write-up
(architecture reasoning, tradeoffs, and a list of real bugs found and fixed
during development). This notebook focuses on the code itself, organized in
the same order as the pipeline runs.

**Usage from the command line** (the required single entry point):
```bash
python main.py --input "Why we procrastinate, and how to stop"
python main.py --input "https://www.ted.com/talks/julian_treasure_how_to_speak_so_that_people_want_to_listen"
```

This notebook mirrors `src/orchestrator.py`, `src/pipeline_stages.py`, and
`src/main.py` — running the cells below reproduces the same pipeline
interactively.


## 1. Setup — imports and environment

In [ ]:
import os
import re
import json
import time
import asyncio
import functools
import hashlib
import logging
import textwrap
import urllib.parse
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import requests
import trafilatura
from youtube_transcript_api import YouTubeTranscriptApi
from dotenv import load_dotenv
from groq import Groq
import edge_tts
from mutagen.mp3 import MP3
from moviepy import (
    VideoFileClip, ImageClip, TextClip, CompositeVideoClip,
    CompositeAudioClip, AudioFileClip, VideoClip, concatenate_videoclips,
)
from moviepy.video.fx import CrossFadeIn, CrossFadeOut, Resize
import numpy as np
from PIL import Image

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
)
logger = logging.getLogger("pipeline")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_ROOT = PROJECT_ROOT / "cache"
ASSETS_DIR = PROJECT_ROOT / "outputs" / "scene_assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

_groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


## 2. Orchestrator — the task graph engine

A small, general-purpose task runner (not tied to video generation at all)
that gives every stage three things the assignment explicitly asks for:

1. **Explicit task boundaries** — a task graph, not a linear script.
2. **Independent retryability** — each task retries up to 3 times with
   linear backoff before failing, and never re-runs tasks that already
   succeeded.
3. **Disk caching keyed by input hash** — re-running the pipeline with the
   same input skips already-completed work entirely.

A hand-rolled orchestrator was chosen over a library like Prefect/Airflow
because the pipeline is a small, fixed, linear graph — a full workflow
engine would add real dependency and learning-curve cost without adding
capability needed at this scope. Every task here is a plain Python function
with an explicit input/output contract, so migrating to a real framework
later would be mechanical, not a redesign.

In [ ]:
class TaskFailedError(RuntimeError):
    """Raised when a task exhausts all of its retries."""


def _stable_hash(obj: Any) -> str:
    """Turn any JSON-serializable object into a short, stable hash string.
    Same inputs -> same hash -> same cache file."""
    payload = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]


@dataclass
class TaskResult:
    """Standard envelope every task returns."""
    name: str
    output: Any
    from_cache: bool
    duration_s: float


class CachedRetryableTask:
    """Wraps a plain function with caching + retry behaviour."""

    def __init__(self, name, func, cache_dir, max_retries=3, backoff_s=2.0):
        self.name = name
        self.func = func
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.max_retries = max_retries
        self.backoff_s = backoff_s

    def _cache_path(self, inputs: dict) -> Path:
        key = _stable_hash(inputs)
        return self.cache_dir / f"{key}.json"

    def run(self, inputs: dict, force: bool = False) -> TaskResult:
        cache_path = self._cache_path(inputs)
        start = time.time()

        if not force and cache_path.exists():
            logger.info("[%s] cache HIT (%s) -- skipping re-run", self.name, cache_path.name)
            cached = json.loads(cache_path.read_text())
            return TaskResult(name=self.name, output=cached, from_cache=True,
                               duration_s=time.time() - start)

        last_exc = None
        for attempt in range(1, self.max_retries + 1):
            try:
                logger.info("[%s] attempt %d/%d ...", self.name, attempt, self.max_retries)
                output = self.func(inputs)
                cache_path.write_text(json.dumps(output, indent=2, default=str))
                logger.info("[%s] succeeded, cached -> %s", self.name, cache_path.name)
                return TaskResult(name=self.name, output=output, from_cache=False,
                                   duration_s=time.time() - start)
            except Exception as exc:
                last_exc = exc
                logger.warning("[%s] attempt %d failed: %s", self.name, attempt, exc)
                if attempt < self.max_retries:
                    delay = self.backoff_s * attempt
                    logger.info("[%s] retrying in %.1fs ...", self.name, delay)
                    time.sleep(delay)

        raise TaskFailedError(f"Task '{self.name}' failed after {self.max_retries} attempts") from last_exc


@dataclass
class Pipeline:
    """Runs a fixed sequence of tasks, passing each task's output forward."""
    tasks: list = field(default_factory=list)

    def add(self, task) -> "Pipeline":
        self.tasks.append(task)
        return self

    def run(self, initial_input: dict, force=None) -> dict:
        force = force or set()
        state = {"initial_input": initial_input}
        results = {}
        for task in self.tasks:
            task_inputs = {**state}
            result = task.run(task_inputs, force=(task.name in force))
            state[task.name] = result.output
            results[task.name] = result
            logger.info("[%s] done in %.2fs (from_cache=%s)", task.name, result.duration_s, result.from_cache)
        return results


def task(name: str, cache_dir: Path, max_retries: int = 3, backoff_s: float = 2.0):
    """Decorator: turns a plain function into a CachedRetryableTask."""
    def decorator(func):
        wrapped = functools.wraps(func)(func)
        return CachedRetryableTask(name=name, func=wrapped, cache_dir=cache_dir,
                                     max_retries=max_retries, backoff_s=backoff_s)
    return decorator


## 3. Stage 1 — `gather_info`

Branches on input type:
- **Plain topic** -> Groq LLM asked to act as a careful researcher.
- **TED talk URL** -> TED's transcript is JavaScript-rendered (confirmed by
  inspecting raw HTML — no transcript text present in the initial response).
  Fixed by scraping the talk's title (reliably present in plain HTML),
  searching YouTube for that title, and fetching the real transcript from
  the matched video.
- **Plain YouTube URL** -> direct transcript fetch.
- **Other article URL** -> `trafilatura` extracts clean article text.

This layered fallback (TED-specific -> YouTube-specific -> generic article)
is deliberately general rather than hardcoded to the two PDF examples.

In [ ]:
def _research_topic(topic: str) -> str:
    response = _groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": (
                "You are a careful researcher preparing background material "
                "for a short 30-40 second video. Provide accurate, well-known, "
                "widely-agreed-upon facts and explanations. Do not invent "
                "statistics or studies. Write 3-5 short paragraphs covering: "
                "what the topic means, why it happens/matters, and any "
                "practical takeaway. Keep it factual and clear."
            )},
            {"role": "user", "content": f"Topic: {topic}"},
        ],
        temperature=0.4,
    )
    return response.choices[0].message.content


def _extract_youtube_video_id(url: str):
    patterns = [
        r"(?:youtube\.com/watch\?v=)([\w-]+)",
        r"(?:youtu\.be/)([\w-]+)",
        r"(?:youtube\.com/embed/)([\w-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def _search_youtube_video_id(query: str):
    """Free, no-API-key YouTube search: hits the public results page and
    pulls the first videoId out of its embedded page data."""
    resp = requests.get(
        "https://www.youtube.com/results",
        params={"search_query": query},
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=15,
    )
    resp.raise_for_status()
    match = re.search(r'"videoId":"([\w-]{11})"', resp.text)
    return match.group(1) if match else None


def _get_page_title(url: str):
    downloaded = trafilatura.fetch_url(url)
    if downloaded is None:
        return None
    metadata = trafilatura.extract_metadata(downloaded)
    return metadata.title if metadata else None


def _fetch_ted_transcript(url: str):
    """TED's transcript is JS-rendered, so we can't scrape it directly.
    Get the title -> search YouTube -> fetch that video's real transcript."""
    if "ted.com/talks" not in url:
        return None
    title = _get_page_title(url)
    if not title:
        return None
    video_id = _search_youtube_video_id(f"{title} TED")
    if not video_id:
        return None
    return _fetch_youtube_transcript(video_id)


def _clean_transcript_text(text: str) -> str:
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _fetch_youtube_transcript(video_id: str) -> str:
    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id)
    full_text = " ".join(snippet.text for snippet in fetched)
    return _clean_transcript_text(full_text)


def _scrape_article(url: str) -> str:
    downloaded = trafilatura.fetch_url(url)
    if downloaded is None:
        raise RuntimeError(f"Could not download page: {url}")
    text = trafilatura.extract(downloaded)
    if not text:
        raise RuntimeError(f"Could not extract article text from: {url}")
    return text


def _gather_from_url(url: str) -> dict:
    ted_transcript = _fetch_ted_transcript(url)
    if ted_transcript:
        return {"source_type": "url", "url_kind": "ted_transcript_via_search", "raw_content": ted_transcript}

    video_id = _extract_youtube_video_id(url)
    if video_id:
        raw_content = _fetch_youtube_transcript(video_id)
        return {"source_type": "url", "url_kind": "video_transcript", "raw_content": raw_content}

    raw_content = _scrape_article(url)
    return {"source_type": "url", "url_kind": "article", "raw_content": raw_content}


@task("gather_info", cache_dir=CACHE_ROOT / "gather_info")
def gather_info(inputs: dict) -> dict:
    user_input = inputs["initial_input"]["query"].strip()
    is_url = user_input.startswith("http://") or user_input.startswith("https://")

    if not is_url:
        raw_content = _research_topic(user_input)
        return {"source_type": "topic", "raw_content": raw_content}

    return _gather_from_url(user_input)


## 4. Stage 2 — `storyboard`

A single structured LLM call performs content **selection** (which points
earn a place in ~35 seconds), **narrative structuring** (beginning / middle
/ end), and generates one **Visual Style Guide** applied to every scene for
cross-scene consistency — one call rather than separate calls, so selection
and phrasing can't drift out of sync.

For URL-sourced input, an explicit grounding instruction is added.
Grounding was verified by manually cross-checking generated narration
against the real source transcript; one early generation included an
ungrounded phrase, which motivated strengthening the prompt with an
explicit self-check instruction.

Scene duration (6.5-6.75s) is enforced by **clamping in code**, not just
requesting it in the prompt — testing showed the LLM frequently returned
durations outside the requested range.

In [ ]:
def _build_storyboard_prompt(raw_content: str, source_type: str, url_kind=None) -> str:
    grounding_rule = ""
    if source_type == "url":
        grounding_rule = (
            "IMPORTANT: This content comes from a real source. Every narration line "
            "must be directly supported by the source text below. Do not invent facts, "
            "statistics, or claims that are not present in the source -- this includes "
            "adding conclusions, themes, or phrases that sound plausible but were not "
            "actually stated (e.g. do not add words like 'connection' or 'trust' as a "
            "takeaway unless the source itself uses that word or a very close paraphrase). "
            "Before finalizing, mentally check each narration line against the source "
            "and remove anything you cannot point to directly in the text.\n\n"
        )

    return f"""{grounding_rule}Source content:
\"\"\"
{raw_content}
\"\"\"

Task: Turn this into a storyboard for a 30-40 second vertical video (9:16, TikTok/Reels-style).

Step 1 - Selection: This source likely contains more material than 35 seconds can hold.
Identify the 4 to 6 most important, most story-worthy points. Prioritize points that
together form a coherent narrative arc (hook -> context/explanation -> resolution/takeaway),
over points that are individually interesting but disconnected.

Step 2 - Storyboard: Turn the selected points into 6 to 7 scenes. HARD CONSTRAINT: every
single scene's duration_s MUST be either exactly 5 or exactly 6 -- no other value is
allowed (not 7, not 8, not 4). This matches what AI video generation models can natively
produce in one call. Each scene needs:
- narration: a natural spoken sentence, slightly fuller/more descriptive than a bare
  headline -- aim for a complete, flowing thought rather than the shortest possible
  phrasing (e.g. prefer \"We've all sat down to work and felt the pull to do anything else\"
  over \"We all procrastinate\"). Conversational tone, contractions welcome.

Example of correctly-paced narration for a 5-second scene (12-13 words):
\"Most of us know that feeling -- staring at a task, unable to start.\"
(NOT \"We procrastinate\" -- too short, leaves dead air when spoken aloud)
- visual: a concrete, filmable description of what should be shown on screen (subject,
  action, setting -- specific enough to generate an image/short clip from)
- duration_s: this scene's length in seconds (all scenes summed should total 30-40s)

Step 3 - Style Guide: Write ONE short visual style description (palette, lighting, subject
consistency, camera feel) that will be applied to every scene's image, so the whole video
looks visually consistent.

Respond with ONLY valid JSON, no other text, in exactly this shape:
{{
  \"style_guide\": \"string describing the consistent visual style\",
  \"scenes\": [
    {{\"scene_id\": 1, \"narration\": \"...\", \"visual\": \"...\", \"duration_s\": 6}},
    {{\"scene_id\": 2, \"narration\": \"...\", \"visual\": \"...\", \"duration_s\": 6}}
  ]
}}"""


def _generate_storyboard_llm(raw_content: str, source_type: str, url_kind=None) -> dict:
    prompt = _build_storyboard_prompt(raw_content, source_type, url_kind)

    response = _groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": (
                "You are a short-form video director. You respond with ONLY valid "
                "JSON matching the exact schema requested. No markdown code fences, "
                "no explanation text before or after -- just the raw JSON object."
            )},
            {"role": "user", "content": prompt},
        ],
        temperature=0.6,
    )

    raw_text = response.choices[0].message.content.strip()
    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]
        raw_text = raw_text.strip()

    return json.loads(raw_text)


@task("storyboard", cache_dir=CACHE_ROOT / "storyboard")
def generate_storyboard(inputs: dict) -> dict:
    gather_info_output = inputs["gather_info"]
    raw_content = gather_info_output["raw_content"]
    source_type = gather_info_output["source_type"]
    url_kind = gather_info_output.get("url_kind")

    storyboard = _generate_storyboard_llm(raw_content, source_type, url_kind)

    # Duration is a constraint WE defined, not a fact the LLM needs to "get
    # right" -- clamp deterministically rather than retrying and hoping.
    for scene in storyboard["scenes"]:
        scene["duration_s"] = min(6, max(5, scene["duration_s"]))

    total_duration = sum(scene["duration_s"] for scene in storyboard["scenes"])
    if not (25 <= total_duration <= 45):
        raise ValueError(
            f"Storyboard duration {total_duration}s is outside acceptable range "
            f"even after clamping -- likely too few/many scenes returned."
        )

    # Loose floor only: catches genuinely degenerate narration. Real pacing
    # is authoritatively determined in Stage 3 by measuring actual TTS audio.
    MIN_WORDS = 5
    for scene in storyboard["scenes"]:
        word_count = len(scene["narration"].split())
        if word_count < MIN_WORDS:
            raise ValueError(
                f"Scene {scene['scene_id']} narration is too sparse: {word_count} words "
                f"(minimum {MIN_WORDS}) -- likely malformed LLM output."
            )

    return storyboard


## 5. Stage 3 — `assets`

Per scene: an image (Pollinations.ai, free, no key) using the visual
description plus the shared style guide text for cross-scene consistency,
and narration audio (`edge-tts`, free, no key). Real audio duration is
measured directly from the rendered file and treated as ground truth —
not the LLM's predicted `duration_s`.

Scene video duration is clamped to a 6.5-6.75s band (tuned against the
actual cross-fade math to reliably land the whole video in the 30-40s
range). If real narration would run longer than its scene's video window,
the audio is regenerated at a proportionally faster speaking rate so
narration always finishes within its own scene, rather than bleeding into
the next scene's transition.

In [ ]:
def _generate_scene_image(visual_description: str, style_guide: str, scene_id: int, run_id: str) -> str:
    full_prompt = f"{visual_description}, {style_guide}"
    encoded_prompt = urllib.parse.quote(full_prompt)
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}"
    params = {"width": 1024, "height": 1820, "nologo": "true"}

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    scene_dir = ASSETS_DIR / run_id
    scene_dir.mkdir(parents=True, exist_ok=True)
    image_path = scene_dir / f"scene_{scene_id}.png"
    image_path.write_bytes(response.content)
    return str(image_path)


async def _generate_scene_audio_async(narration_text: str, output_path: Path, rate: str = "+0%") -> None:
    communicator = edge_tts.Communicate(narration_text, voice="en-US-AriaNeural", rate=rate)
    await communicator.save(str(output_path))


def _generate_scene_audio(narration_text: str, scene_id: int, run_id: str, rate: str = "+0%") -> str:
    scene_dir = ASSETS_DIR / run_id
    scene_dir.mkdir(parents=True, exist_ok=True)
    audio_path = scene_dir / f"scene_{scene_id}.mp3"
    asyncio.run(_generate_scene_audio_async(narration_text, audio_path, rate))
    return str(audio_path)


def _get_audio_duration(audio_path: str) -> float:
    """Real, measured narration duration -- ground truth for scene timing."""
    audio = MP3(audio_path)
    return audio.info.length


MIN_SCENE_DURATION_S = 6.5
MAX_SCENE_DURATION_S = 6.75  # keeps a 6-scene total under the 40s ceiling


def _generate_assets_for_scene(scene, style_guide, run_id, scene_dir):
    image_path = _generate_scene_image(scene["visual"], style_guide, scene["scene_id"], run_id)
    audio_path = _generate_scene_audio(scene["narration"], scene["scene_id"], run_id)
    real_duration = _get_audio_duration(audio_path)

    video_duration = min(max(real_duration, MIN_SCENE_DURATION_S), MAX_SCENE_DURATION_S)

    # If narration would run longer than its scene's video window, speak it
    # slightly faster instead of letting it bleed into the next transition.
    if real_duration > video_duration:
        speedup_factor = real_duration / video_duration
        rate_str = f"+{int((speedup_factor - 1) * 100)}%"
        audio_path = _generate_scene_audio(scene["narration"], scene["scene_id"], run_id, rate=rate_str)
        real_duration = _get_audio_duration(audio_path)

    motion_path = str(scene_dir / f"scene_{scene['scene_id']}_motion.mp4")
    _apply_ken_burns(image_path, video_duration, motion_path)

    return {
        "scene_id": scene["scene_id"],
        "narration": scene["narration"],
        "image_path": image_path,
        "audio_path": audio_path,
        "motion_path": motion_path,
        "planned_duration_s": scene["duration_s"],
        "actual_duration_s": round(video_duration, 2),
        "narration_duration_s": round(real_duration, 2),
    }


@task("assets", cache_dir=CACHE_ROOT / "assets")
def generate_assets(inputs: dict) -> dict:
    scenes = inputs["storyboard"]["scenes"]
    style_guide = inputs["storyboard"]["style_guide"]
    run_id = inputs["initial_input"].get("run_id", "default")

    scene_dir = ASSETS_DIR / run_id
    scene_dir.mkdir(parents=True, exist_ok=True)

    scene_assets = [
        _generate_assets_for_scene(scene, style_guide, run_id, scene_dir)
        for scene in scenes
    ]
    return {"scene_assets": scene_assets}


## 6. Stage 4 — `assembly`

Each scene's still image becomes a moving clip via a **Ken Burns pan/zoom**
effect. An early version resized the clip frame directly over time, which
corrupted the video -- video codecs require every frame to be an identical
size. Fixed by rendering into a fixed-size output canvas and cropping a
growing virtual zoom window back down to constant dimensions on every
frame, and enforcing even width/height (a `libx264` requirement).

Clips are joined with 0.3s cross-fade transitions (reduced from an initial
0.5s, which produced visible "ghosting" between visually different
scenes). Narration audio and captions are both positioned using one shared
timeline calculation, so they cannot drift out of sync with each other.
The final output is scaled and padded (not stretched) to exactly
1080x1920.

In [ ]:
def _apply_ken_burns(image_path: str, duration_s: float, output_path: str) -> str:
    """Slow zoom across a still image, keeping frame dimensions constant
    throughout (a naive resize-per-frame approach corrupts the video, since
    codecs require every frame to be the exact same size)."""
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    w -= w % 2  # libx264 requires even width/height
    h -= h % 2
    img = img.crop((0, 0, w, h))
    img_arr = np.array(img)

    def make_frame(t):
        scale = 1 + 0.04 * (t / duration_s)  # 1.0x -> 1.04x zoom
        new_w, new_h = int(w * scale), int(h * scale)
        resized = np.array(Image.fromarray(img_arr).resize((new_w, new_h)))
        x0 = (new_w - w) // 2
        y0 = (new_h - h) // 2
        return resized[y0:y0 + h, x0:x0 + w]

    clip = VideoClip(make_frame, duration=duration_s)
    clip.write_videofile(output_path, fps=24, codec="libx264", audio=False, logger=None)
    return output_path


CROSSFADE_S = 0.3
TARGET_W, TARGET_H = 1080, 1920


def _compute_scene_timeline(scene_assets: list) -> list:
    """Computes each scene's start time in the final timeline, accounting
    for cross-fade overlaps eating into the naive cumulative sum."""
    timeline = []
    cursor = 0.0
    for i, scene in enumerate(scene_assets):
        start = cursor
        duration = scene["actual_duration_s"]
        timeline.append({"scene_id": scene["scene_id"], "start": start, "duration": duration})
        overlap = CROSSFADE_S if i < len(scene_assets) - 1 else 0.0
        cursor += duration - overlap
    return timeline


def _concatenate_with_crossfades(scene_assets: list, output_path: str) -> float:
    clips = []
    for i, scene in enumerate(scene_assets):
        clip = VideoFileClip(scene["motion_path"])
        if i > 0:
            clip = clip.with_effects([CrossFadeIn(CROSSFADE_S)])
        if i < len(scene_assets) - 1:
            clip = clip.with_effects([CrossFadeOut(CROSSFADE_S)])
        clips.append(clip)

    final = concatenate_videoclips(clips, method="compose", padding=-CROSSFADE_S)
    final.write_videofile(output_path, fps=24, codec="libx264", audio=False, logger=None)

    total_duration = final.duration
    for c in clips:
        c.close()
    final.close()
    return total_duration


def _build_audio_track(scene_assets: list, timeline: list, total_duration: float) -> CompositeAudioClip:
    audio_clips = []
    for scene, t in zip(scene_assets, timeline):
        audio = AudioFileClip(scene["audio_path"]).with_start(t["start"])
        audio_clips.append(audio)
    return CompositeAudioClip(audio_clips).with_duration(total_duration)


def _build_caption_clips(scene_assets: list, timeline: list) -> list:
    caption_clips = []
    for scene, t in zip(scene_assets, timeline):
        caption = (
            TextClip(
                text=scene["narration"], font_size=40, color="white",
                stroke_color="black", stroke_width=2, method="caption",
                size=(int(TARGET_W * 0.85), None), text_align="center",
            )
            .with_start(t["start"]).with_duration(t["duration"])
            .with_position(("center", 0.78), relative=True)
        )
        caption_clips.append(caption)
    return caption_clips


def _resize_and_pad(clip):
    """Scales to fit within 1080x1920 preserving aspect ratio, then pads
    with black bars -- avoids distortion since source images aren't
    natively 9:16."""
    scale = min(TARGET_W / clip.w, TARGET_H / clip.h)
    resized = clip.with_effects([Resize(scale)])
    return CompositeVideoClip([resized.with_position("center")], size=(TARGET_W, TARGET_H))


def assemble_final_video(scene_assets: list, run_id: str) -> dict:
    output_dir = CACHE_ROOT.parent / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    final_path = str(output_dir / f"final_{run_id}.mp4")
    silent_path = str(output_dir / f"_silent_{run_id}.mp4")

    timeline = _compute_scene_timeline(scene_assets)
    total_duration = _concatenate_with_crossfades(scene_assets, silent_path)

    audio_track = _build_audio_track(scene_assets, timeline, total_duration)
    caption_clips = _build_caption_clips(scene_assets, timeline)

    base_video = VideoFileClip(silent_path)
    base_video = _resize_and_pad(base_video)

    video = CompositeVideoClip([base_video, *caption_clips], size=(TARGET_W, TARGET_H)).with_audio(audio_track)
    video.write_videofile(final_path, fps=24, codec="libx264", audio_codec="aac", logger=None)

    video.close()
    base_video.close()
    audio_track.close()

    return {"final_video_path": final_path, "duration_s": round(total_duration, 2)}


@task("assembly", cache_dir=CACHE_ROOT / "final")
def assemble_video(inputs: dict) -> dict:
    scene_assets = inputs["assets"]["scene_assets"]
    run_id = inputs["initial_input"].get("run_id", "default")
    return assemble_final_video(scene_assets, run_id)


## 7. Building and running the pipeline

Wires the four tasks together in order and exposes the single entry point.
(`main.py` provides the same thing as a command-line script; this cell is
the interactive/notebook equivalent.)

In [ ]:
def build_pipeline() -> Pipeline:
    pipeline = Pipeline()
    pipeline.add(gather_info)
    pipeline.add(generate_storyboard)
    pipeline.add(generate_assets)
    pipeline.add(assemble_video)
    return pipeline


def run_pipeline(user_input: str, force=None):
    run_id = hashlib.sha256(user_input.encode()).hexdigest()[:10]
    pipeline = build_pipeline()
    results = pipeline.run(
        initial_input={"query": user_input, "run_id": run_id},
        force=force or set(),
    )
    print("\n=== PIPELINE RUN SUMMARY ===")
    for name, result in results.items():
        print(f"  {name:12s} | cached={result.from_cache!s:5s} | {result.duration_s:.2f}s")
    final = results["assembly"].output
    print(f"\nFinal video: {final['final_video_path']} ({final['duration_s']}s)")
    return results


## 8. Example runs — the two required inputs

**Input Set 1 (topic-only):**

In [ ]:
results_topic = run_pipeline("Why we procrastinate, and how to stop")


**Input Set 2 (source URL — TED talk):**

In [ ]:
results_url = run_pipeline(
    "https://www.ted.com/talks/julian_treasure_how_to_speak_so_that_people_want_to_listen"
)


## 9. Notes on reproducibility

- Requires a free Groq API key (`GROQ_API_KEY`) in a `.env` file at the
  project root. No other API keys are needed — Pollinations.ai and
  `edge-tts` require no authentication.
- First runs take several minutes (real image/audio/video generation per
  scene). Re-running with identical input hits the cache and completes in
  under a second per stage — demonstrated by running the cells above twice.
- See `README.md` for full setup instructions and dependency versions.
- See `APPROACH.md` for the full design write-up, including a list of real
  bugs found and fixed during development.